In [1]:
import torch
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader
import torch.mps as mps  # MPS for Mac acceleration
import os
from torchvision.models import resnet18, ResNet18_Weights
import numpy as np

image_path = '../train_test_images'

In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

mps


In [3]:
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.RandomAffine(degrees=0, shear=10),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # mean?
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_data = datasets.ImageFolder(os.path.join(image_path, 'train'), transform=train_transforms)
val_data = datasets.ImageFolder(os.path.join(image_path, 'val'), transform=val_transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

In [4]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 3)

model = model.to(device)


criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [5]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler=None, epochs=10, grad_clip=None):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct, total = 0, 0
        running_precision, running_recall = 0.0, 0.0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()

            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            precision = (predicted == labels).float().mean().item()
            recall = correct / total
            running_precision += precision
            running_recall += recall

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = correct / total
        epoch_precision = running_precision / len(train_loader)
        epoch_recall = running_recall / len(train_loader)
        
        if scheduler:
            scheduler.step()

        print(f'Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}, Precision: {epoch_precision:.4f}, Recall: {epoch_recall:.4f}')

        model.eval()
        val_loss = 0.0
        correct_val, total_val = 0, 0
        val_precision, val_recall = 0.0, 0.0

        with torch.no_grad():
            for batch_idx, (inputs, labels) in enumerate(val_loader):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                correct_val += (predicted == labels).sum().item()
                total_val += labels.size(0)

                val_precision += (predicted == labels).float().mean().item()
                val_recall += correct_val / total_val

        val_loss /= len(val_loader)
        val_acc = correct_val / total_val
        val_precision /= len(val_loader)
        val_recall /= len(val_loader)

        print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}, Validation Precision: {val_precision:.4f}, Validation Recall: {val_recall:.4f}')


In [6]:
# Run training
train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10)

Epoch [1/10], Loss: 0.8300, Accuracy: 0.7131, Precision: 0.7151, Recall: 0.6167
Validation Loss: 2.0551, Validation Accuracy: 0.2712, Validation Precision: 0.2500, Validation Recall: 0.5066
Epoch [2/10], Loss: 0.5136, Accuracy: 0.8308, Precision: 0.8283, Recall: 0.8323
Validation Loss: 2.1607, Validation Accuracy: 0.4831, Validation Precision: 0.4453, Validation Recall: 0.7184
Epoch [3/10], Loss: 0.3877, Accuracy: 0.8672, Precision: 0.8651, Recall: 0.8423
Validation Loss: 0.4495, Validation Accuracy: 0.8644, Validation Precision: 0.8750, Validation Recall: 0.8268
Epoch [4/10], Loss: 0.4156, Accuracy: 0.8630, Precision: 0.8638, Recall: 0.8424
Validation Loss: 2.8873, Validation Accuracy: 0.4407, Validation Precision: 0.4560, Validation Recall: 0.5607
Epoch [5/10], Loss: 0.2933, Accuracy: 0.8929, Precision: 0.8958, Recall: 0.9128
Validation Loss: 0.5439, Validation Accuracy: 0.8729, Validation Precision: 0.8757, Validation Recall: 0.8484
Epoch [6/10], Loss: 0.3093, Accuracy: 0.8994, Prec